In [27]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

from openai import AsyncOpenAI
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel, input_guardrail, GuardrailFunctionOutput
import asyncio
import os
import sendgrid
from sendgrid import SendGridAPIClient
from sendgrid.helpers.mail import Mail, Email, To, Content
from pydantic import BaseModel
from agents import handoff

## EXPERIMENTING TOOLS

In [2]:
@function_tool
def add(num1: int, num2: int, num3: int=0):
    """adds two or three numbers and retuns the result"""
    return num1 + num2 + num3

tools = [add]

In [3]:
groq_client = AsyncOpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.getenv("OPENROUTER_API_KEY"))
llama = OpenAIChatCompletionsModel(model="openai/gpt-4o-mini", openai_client=groq_client)


agent1 = Agent(name="Mathematician", instructions="you are a mathematician who can do simple and complex math. Use the add tool to add upto three numbers you can do a sequence of calls to add up more than 3 numbers.", model=llama, tools=tools)

In [4]:
with trace("Math_trace"):
    result = await Runner.run(agent1, "What is 2 plus 4 plus 7 plus 10")
print(result.final_output)

The sum of 2 plus 4 plus 7 is 13. Adding 10 to that gives a total of 23. 

So, 2 + 4 + 7 + 10 = 23.


## Using agent as a tool

In [5]:
agent1 = agent1.as_tool(tool_name="adder", tool_description="Adds two or more numbers together and gives you an answer")
tools1=[agent1]

In [6]:
agent2 = Agent(name="QA", instructions="You are a question answering agent who answers users questions. Use the add tool to add upto three numbers you can do a sequence of calls to add up more than 3 numbers.", model=llama, tools=tools1)
agent1

FunctionTool(name='adder', description='Adds two or more numbers together and gives you an answer', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x7e91820a5010>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False)

In [7]:
with trace("Math_trace"):
    result = await Runner.run(agent2, "What is 2 plus 4 plus 7 plus 10")
print(result.final_output)

The sum of 2, 4, 7, and 10 is 23.


## EXPERIMENTING SENDGRID

In [8]:
def send_test_email():
    message = Mail(
        from_email='eachala5@gmail.com',
        to_emails='eachala5@gmail.com',
        subject='Test Me',
        html_content='<strong>It worked</strong>')

    sg = SendGridAPIClient(os.getenv('SENDGRID_API_KEY'))
    response = sg.send(message)
    print(response.status_code)

In [115]:
send_test_email()

202


## RUNNING AGENTS IN PARALLEL

In [9]:
instructions1 = "You are a sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold emails."

instructions2 = "You are a humorous, engaging sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write witty, engaging cold emails that are likely to get a response."

instructions3 = "You are a busy sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write concise, to the point cold emails."

In [10]:
sales_agent1 = Agent(
        name="Professional Sales Agent",
        instructions=instructions1,
        model=llama
)

sales_agent2 = Agent(
        name="Engaging Sales Agent",
        instructions=instructions2,
        model=llama
)

sales_agent3 = Agent(
        name="Busy Sales Agent",
        instructions=instructions3,
        model=llama
)

In [31]:
message = "Write a cold sales email"

with trace("Parallel cold emails"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )

outputs = [result.final_output for result in results]

for output in outputs:
    print(output + "\n\n")


Subject: Enhance Your SOC2 Compliance with AI-Driven Expertise

Dear [Recipient's Name],

I hope this email finds you well. As a [Recipient's Job Title] at [Company Name], I'm sure you understand the importance of maintaining the highest standards of security and compliance in today's digital landscape. With the ever-evolving threat landscape and increasing regulatory requirements, ensuring SOC2 compliance is no longer just a best practice, but a business imperative.

At ComplAI, we recognize the challenges and complexities associated with SOC2 compliance and audit preparation. That's why we've developed an innovative SaaS tool, powered by artificial intelligence, to streamline and simplify the compliance process. Our solution provides real-time monitoring, automated risk assessment, and guided remediation, ensuring your organization stays ahead of potential vulnerabilities and audit requirements.

By leveraging our AI-driven expertise, you can:

* Reduce the burden of manual complianc

OPENAI_API_KEY is not set, skipping trace export


## SALES MANAGER (USING TOOLS APPROACH)

In [11]:
@function_tool
def send_email(body: str):
    message = Mail(
        from_email='eachala5@gmail.com',
        to_emails='eachala5@gmail.com',
        subject='Test Me',
        #html_content=body,
        plain_text_content=body
        )

    sg = SendGridAPIClient(os.getenv('SENDGRID_API_KEY'))
    response = sg.send(message)

    return {"status": "success"}

In [12]:
description = "Write a cold sales email"
tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent1.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent1.as_tool(tool_name="sales_agent3", tool_description=description)

tools=[tool1, tool2, tool3, send_email]
tools

[FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x7e91820a6d80>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False),
 FunctionTool(name='sales_agent2', description='Write a cold sales email', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._Fa

In [13]:
instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
 
3. Use the send_email tool to send the best email (and only the best email) to the user.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must send ONE email using the send_email tool — never more than one.
"""


sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, model=llama)

message = "Send a cold sales email addressed to 'Dear CEO'"

with trace("Sales manager"):
    result = await Runner.run(sales_manager, message)

## SALES MANAGER(USING HANDOFFS APPROACH)

In [14]:

subject_instructions = "You can write a subject for a cold sales email. \
You are given a message and you need to write a subject for an email that is likely to get a response."

html_instructions = "You can convert a text email body to an HTML email body. \
You are given a text email body which might have some markdown \
and you need to convert it to an HTML email body with simple, clear, compelling layout and design."

subject_writer = Agent(name="Email subject writer", instructions=subject_instructions, model=llama)
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject for a cold sales email")

html_converter = Agent(name="HTML email body converter", instructions=html_instructions, model=llama)
html_tool = html_converter.as_tool(tool_name="html_converter",tool_description="Convert a text email body to an HTML email body")


In [22]:
@function_tool
def send_html_email(subject: str, html_body: str):
    """ Send out an email with the given subject and HTML body to all sales prospects """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("eachala5@gmail.com")
    to_email = To("eachala5@gmail.com")
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    print("Tooollll send_html_email called")
    return {"status": "success"}

In [23]:
emailer_tools = [subject_tool, html_tool, send_html_email]
emailer_tools

[FunctionTool(name='subject_writer', description='Write a subject for a cold sales email', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x7e9180f24fe0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False),
 FunctionTool(name='html_converter', description='Convert a text email body to an HTML email body', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties'

In [24]:
instructions ="You are an email formatter and sender. You receive the body of an email to be sent. \
You first use the subject_writer tool to write a subject for the email, then use the html_converter tool to convert the body to HTML. \
Finally, you use the send_html_email tool to send the email with the subject and HTML body(Again use send_html_email to send the email with the subject and HTML body)."

class EmailInput(BaseModel):
    email_body: str


emailer_agent = Agent(
    name="Email Manager",
    instructions=instructions,
    tools=emailer_tools,
    model=llama,
    handoff_description="Convert an email to HTML and send it")


In [25]:
tools = [tool1, tool2, tool3]
handoffs = [emailer_agent]
# def on_email_handoff(agent, input: EmailInput):
#     print(f"Handing off email: {input.email_body}")

# handoffs = [handoff(
#     emailer_agent,
#     input_type=EmailInput,
#     on_handoff=on_email_handoff
# )]

print(tools)
print(handoffs)

[FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x7e91820a6d80>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False), FunctionTool(name='sales_agent2', description='Write a cold sales email', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._Fai

In [28]:
sales_manager_instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
You can use the tools multiple times if you're not satisfied with the results from the first try.
 
3. Handoff for Sending: Pass ONLY the winning email draft to the 'Email Manager' agent. The Email Manager will take care of formatting and sending.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must hand off exactly ONE email to the Email Manager — never more than one.
"""


sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=tools,
    handoffs=handoffs,
    model=llama)

message = "Send out a cold sales email addressed to Dear CEO from Alice"

with trace("Automated SDR"):
    result = await Runner.run(sales_manager, message)

Tooollll send_html_email called
